# Testing Django Models and ORM

## Database Operations in Tests
---

In this lesson, you'll learn how to **test Django models and ORM operations** — the data layer of your application.

You've tested views. Now you'll focus on models: validation, querysets, and relationships.

Examples use **`mysite/blog`** (`Blog`, `Comment`, `BlogReview`, `User`). Run `pytest` from the directory with `manage.py` (lessons 15a–15b).


1. [The django_db Marker](#The-django_db-Marker),
    - [Why You Need It](#Why-You-Need-It),
    - [Marker vs db Fixture](#Marker-vs-db-Fixture),
    - [Transaction Behavior](#Transaction-Behavior),
2. [Creating Test Data](#Creating-Test-Data),
    - [Direct Object Creation](#Direct-Object-Creation),
    - [Fixtures for Test Data](#Fixtures-for-Test-Data),
    - [Fixture Composition](#Fixture-Composition),
3. [Testing Model Methods](#Testing-Model-Methods),
    - [Testing __str__](#Testing-__str__),
    - [Testing derived values](#Testing-derived-values),
    - [Testing related rows](#Testing-related-rows),
4. [Testing Model Validation](#Testing-Model-Validation),
    - [Testing Field Constraints](#Testing-Field-Constraints),
    - [Testing clean() Method](#Testing-clean()-Method),
    - [Testing Validators](#Testing-Validators),
5. [Testing QuerySets](#Testing-QuerySets),
    - [Testing Filter Results](#Testing-Filter-Results),
    - [Reusable query patterns](#Reusable-query-patterns),
    - [Testing Aggregations](#Testing-Aggregations),
6. [Testing Relationships](#Testing-Relationships),
    - [Testing ForeignKey](#Testing-ForeignKey),
    - [Testing Reverse Relations](#Testing-Reverse-Relations),
    - [Testing Cascade Delete](#Testing-Cascade-Delete),
7. [Practical Example: Testing blog models](#Practical-Example:-Testing-blog-models),
8. [🧠 EXERCISE 🧠, Test Blog __str__ and ordering](#🧠-EXERCISE-🧠,-Test-Blog-__str__-and-ordering),
9. [🧠 EXERCISE 🧠, Test QuerySets on Blog](#🧠-EXERCISE-🧠,-Test-QuerySets-on-Blog).

<br>


## The `django_db` Marker

---

### Why You Need It

---

By default, pytest-django **blocks database access** in tests. This is intentional - it prevents accidental database operations in tests that don't need them.

Without the marker:

In [ ]:
# This will FAIL (no django_db)

from datetime import date
from blog.models import Blog


def test_without_db_access():
    Blog.objects.create(
        title="Test",
        author="alice",
        published_date=date(2026, 1, 1),
    )


**Error:**
```
RuntimeError: Database access not allowed, use the "django_db" mark, 
or the "db" or "transactional_db" fixtures to enable it.
```

To enable database access, use `@pytest.mark.django_db`:

In [ ]:
import pytest
from datetime import date

from blog.models import Blog


@pytest.mark.django_db
def test_with_db_access():
    blog = Blog.objects.create(
        title="Hello",
        author="alice",
        published_date=date(2026, 1, 1),
    )
    assert blog.pk is not None


<br>

### Marker vs db Fixture

---

There are two ways to enable database access:

**1. The `@pytest.mark.django_db` marker:**

In [ ]:
@pytest.mark.django_db
def test_something():
    # Database access enabled
    pass

**2. The `db` fixture:**

In [ ]:
def test_something(db):
    # Database access enabled
    pass

| Approach | Use Case |
|----------|----------|
| `@pytest.mark.django_db` | Test function needs database directly |
| `db` fixture | Fixture needs database, test uses fixture |

<br>

**Best practice:** Use the `db` fixture in your data fixtures, then tests automatically get database access:

In [ ]:
@pytest.fixture
def blog(db):  # db fixture here
    from datetime import date
    from blog.models import Blog

    return Blog.objects.create(
        title="Hello",
        author="alice",
        published_date=date(2026, 2, 1),
    )


def test_blog_str(blog):  # No marker needed — db via fixture
    assert str(blog) == "Hello"

<br>

### Transaction Behavior

---

<br>


<br>

For tests that need **real transactions** (e.g., testing transaction behavior):

In [ ]:
@pytest.mark.django_db(transaction=True)
def test_with_real_transactions():
    """This test uses real transactions, not wrapped in a rollback."""
    pass

By default, each test runs in a **database transaction** that gets rolled back:

```
Test 1:
  BEGIN TRANSACTION
  → Create Blog (id=1)
  → Assert something
  ROLLBACK

Test 2:
  BEGIN TRANSACTION
  → Table is empty again!
  → Create Comment
  ROLLBACK
```

<br>

**Benefits:**
- Tests are isolated - don't affect each other
- No cleanup needed
- Fast - rollback is cheaper than delete


### Direct Object Creation

---

The simplest way - create objects directly in the test:

In [ ]:
import pytest
from datetime import date

from blog.models import Blog, Comment


@pytest.mark.django_db
def test_blog_and_comment_creation():
    blog = Blog.objects.create(
        title="ORM tips",
        author="bob",
        published_date=date(2026, 3, 15),
    )
    comment = Comment.objects.create(blog=blog, content="one two three four")
    assert comment.pk is not None
    assert comment.blog_id == blog.pk


**Problem:** This is verbose and repeated across tests.

<br>

### Fixtures for Test Data

---

Better approach — use **fixtures** for reusable test data:


In [ ]:
# blog/tests/conftest.py

import pytest
from datetime import date

from blog.models import Blog, Comment, BlogReview
from django.contrib.auth import get_user_model

User = get_user_model()


@pytest.fixture
def blog(db):
    return Blog.objects.create(
        title="Refactoring",
        author="martinfowler",
        published_date=date(2026, 1, 10),
    )


@pytest.fixture
def user(db):
    return User.objects.create_user(
        username="alice",
        email="alice@example.com",
        password="secret123",
    )


@pytest.fixture
def comment(blog):
    return Comment.objects.create(blog=blog, content="one two three four")


@pytest.fixture
def review(blog, user):
    return BlogReview.objects.create(blog=blog, user=user, rating=5, comment="Great")


Now tests are clean:

In [ ]:
def test_blog_str(blog):
    assert str(blog) == "Refactoring"


def test_comment_linked_to_blog(comment, blog):
    assert comment.blog == blog


<br>

### Fixture Composition

---

<br>

### Testing __str__

---

Always test `__str__` — it is used in admin, the shell, and debugging:


In [ ]:
import pytest
from django.contrib.auth import get_user_model

from blog.models import BlogReview


@pytest.fixture
def blog_with_reviews(blog):
    User = get_user_model()
    u1 = User.objects.create_user("u1", "u1@e.com", "x")
    u2 = User.objects.create_user("u2", "u2@e.com", "x")
    BlogReview.objects.create(blog=blog, user=u1, rating=5, comment="A")
    BlogReview.objects.create(blog=blog, user=u2, rating=3, comment="B")
    return blog


def test_blog_review_count(blog_with_reviews):
    assert blog_with_reviews.reviews.count() == 2


<br>

## Testing Model Methods

---

<br>

### Testing derived values

---

The course `Blog` model has no business `@property` helpers; the next cell tests **defaults** and **`CategoryType`** on real fields.


<br>


In [ ]:
class TestBlogStr:
    def test_str_returns_title(self, blog):
        assert str(blog) == "Refactoring"

    def test_str_unicode_title(self, db):
        from datetime import date
        from blog.models import Blog

        b = Blog.objects.create(
            title="Contribution",
            author="janedoe",
            published_date=date(2026, 4, 1),
        )
        assert str(b) == "Contribution"


<br>

### Testing related rows

---

Models often expose behaviour through **related managers** (`blog.comments`, `blog.reviews`, …):


In [ ]:
# The course Blog model has no helper @property; test defaults and fields.

from blog.models import Blog, CategoryType


class TestBlogFields:
    def test_default_category_is_tech(self, blog):
        assert blog.category_type == CategoryType.TECH

    def test_can_set_category(self, db):
        from datetime import date

        b = Blog.objects.create(
            title="Biz",
            author="corp",
            published_date=date(2026, 1, 1),
            category_type=CategoryType.BUSINESS,
        )
        assert b.category_type == CategoryType.BUSINESS


<br>

<br>

In [ ]:
# Blog has no discount helpers; exercise the ORM via related_name.

from blog.models import Comment


class TestBlogComments:
    def test_starts_with_no_comments(self, blog):
        assert blog.comments.count() == 0

    def test_add_comment(self, blog):
        Comment.objects.create(blog=blog, content="one two three four")
        assert blog.comments.count() == 1

    def test_comment_str(self, comment, blog):
        assert "Comment on" in str(comment)
        assert blog.title in str(comment)


<br>

## Testing Model Validation

---

### Testing Field Constraints

---

Test that database constraints are enforced:

In [ ]:
import pytest
from datetime import date
from django.db import IntegrityError

from blog.models import Blog, BlogReview


class TestBlogReviewConstraints:
    def test_unique_review_per_user(self, blog, django_user_model):
        user = django_user_model.objects.create_user("u", "u@e.com", "pw")
        BlogReview.objects.create(blog=blog, user=user, rating=5)
        with pytest.raises(IntegrityError):
            BlogReview.objects.create(blog=blog, user=user, rating=3)

    def test_published_date_required(self, db):
        with pytest.raises(IntegrityError):
            Blog.objects.create(title="x", author="a", published_date=None)


Trying to add a second row with the exact same blog and user violates the unique constraint at the database level. Because of this, Django—or the backend—will throw an IntegrityError.

<br>

### Testing `clean()` Method

---


Django models can define `clean()` for validation. The course **`Blog` model does not override `clean()`**; much of the cross-field logic lives on **`BlogModelForm`** (`blog/forms.py`).

You can still test **field-level** rules on the model instance with `full_clean()` (this is **not** called automatically by `save()`).


In [ ]:
import pytest
from django.core.exceptions import ValidationError
from datetime import date

from blog.models import Blog


class TestBlogFieldValidation:
    def test_title_max_length(self, db):
        blog = Blog(
            title="x" * 201,
            author="a",
            published_date=date(2026, 1, 1),
        )
        with pytest.raises(ValidationError):
            blog.full_clean()

    def test_valid_blog_full_clean(self, blog):
        blog.full_clean()

    def test_invalid_slug(self, db):
        blog = Blog(
            title="t",
            author="a",
            published_date=date(2026, 1, 1),
            slug="not a slug",
        )
        with pytest.raises(ValidationError):
            blog.full_clean()


**Note:** Use `full_clean()` to trigger model validation. `save()` does NOT call `full_clean()` automatically!

<br>

### Testing Validators

---

In [ ]:
import pytest
from django.core.exceptions import ValidationError

from blog.models import BlogReview


class TestBlogReviewFieldValidation:
    def test_negative_rating_invalid(self, blog, django_user_model):
        user = django_user_model.objects.create_user("r", "r@e.com", "x")
        r = BlogReview(blog=blog, user=user, rating=-1, comment="nope")
        with pytest.raises(ValidationError):
            r.full_clean()

    @pytest.mark.parametrize("rating", [1, 2, 3, 4, 5])
    def test_rating_ok(self, blog, django_user_model, rating):
        user = django_user_model.objects.create_user(f"u{rating}", f"u{rating}@e.com", "x")
        r = BlogReview(blog=blog, user=user, rating=rating, comment="ok")
        r.full_clean()


<br>

## Testing QuerySets

---

### Testing Filter Results

---

In [ ]:
import pytest
from datetime import date

from blog.models import Blog, CategoryType


@pytest.fixture
def multiple_blogs(db):
    return [
        Blog.objects.create(
            title="Alpha",
            author="alice",
            published_date=date(2026, 1, 1),
            category_type=CategoryType.TECH,
        ),
        Blog.objects.create(
            title="Beta",
            author="alice",
            published_date=date(2026, 6, 1),
            category_type=CategoryType.BUSINESS,
        ),
        Blog.objects.create(
            title="Gamma",
            author="bob",
            published_date=date(2024, 6, 1),
            category_type=CategoryType.TECH,
        ),
    ]


class TestBlogQueries:
    def test_filter_by_category(self, multiple_blogs):
        tech = Blog.objects.filter(category_type=CategoryType.TECH)
        assert tech.count() == 2

    def test_filter_by_author(self, multiple_blogs):
        qs = Blog.objects.filter(author="alice")
        assert qs.count() == 2

    def test_default_ordering_newest_first(self, multiple_blogs):
        titles = list(Blog.objects.values_list("title", flat=True))
        assert titles[0] == "Beta"
        assert titles[-1] == "Gamma"


<br>

### Reusable query patterns

---

`Blog` uses the default manager. The same testing approach applies if you later add `Blog.objects.published()` via a custom `Manager`.

The next cell wraps `Blog.objects.filter(...)` in a small helper — **illustrative only**; you do not need to change `models.py` for this lesson.


In [ ]:
import pytest

from blog.models import Blog, CategoryType


def tech_blogs():
    return Blog.objects.filter(category_type=CategoryType.TECH)


class TestBlogQueryPatterns:
    def test_tech_filter(self, multiple_blogs):
        assert tech_blogs().count() == 2

    def test_chain_order_title(self, multiple_blogs):
        titles = list(Blog.objects.filter(author="alice").order_by("title").values_list("title", flat=True))
        assert titles == ["Alpha", "Beta"]


<br>

### Testing Aggregations

---

In [ ]:
from django.db.models import Avg, Count

import pytest

from blog.models import BlogReview, Comment


class TestBlogAggregations:
    def test_average_review_rating(self, blog_with_reviews):
        avg = BlogReview.objects.filter(blog=blog_with_reviews).aggregate(m=Avg("rating"))["m"]
        assert avg == pytest.approx(4.0)

    def test_comment_count(self, blog, comment):
        total = Comment.objects.aggregate(n=Count("id"))["n"]
        assert total >= 1


<br>

## Testing Relationships

---

### Testing ForeignKey

---

In [ ]:
class TestBlogFk:
    def test_comment_has_blog_fk(self, comment, blog):
        assert comment.blog == blog
        assert comment.blog_id == blog.pk

    def test_review_has_user(self, review, user):
        assert review.user == user

    def test_blog_title_from_comment(self, comment):
        assert comment.blog.title == "Refactoring"


<br>

### Testing Reverse Relations

---

In [ ]:
class TestReverseRelations:
    def test_blog_has_reviews(self, blog, review):
        assert review in blog.reviews.all()

    def test_user_has_reviews(self, user, review):
        assert review in user.blog_reviews.all()

    def test_blog_without_reviews(self, db):
        from datetime import date
        from blog.models import Blog

        b = Blog.objects.create(title="Solo", author="solo", published_date=date(2026, 1, 1))
        assert b.reviews.count() == 0


<br>

### Testing Cascade Delete

These examples expect **`blog`**, **`user`**, **`comment`**, and **`review`** fixtures in `blog/tests/conftest.py` (same shapes as in the fixtures section earlier in this notebook).

---

In [ ]:
import pytest


@pytest.mark.django_db
class TestCascadeDelete:
    def test_delete_blog_removes_comments(self, blog, comment):
        cid = comment.pk
        from blog.models import Comment

        blog.delete()
        assert not Comment.objects.filter(pk=cid).exists()

    def test_delete_blog_removes_reviews(self, blog, review):
        rid = review.pk
        from blog.models import BlogReview

        blog.delete()
        assert not BlogReview.objects.filter(pk=rid).exists()

    def test_delete_user_removes_reviews_not_blog(self, blog, review):
        bid = blog.pk
        review.user.delete()
        from blog.models import Blog

        assert Blog.objects.filter(pk=bid).exists()


<br>

## Practical Example: Testing blog models

---


Let's put it together with a focused suite for **`Blog`**, **`Comment`**, and **`BlogReview`** (`mysite/blog/models.py`).


In [ ]:
# blog/tests/conftest.py

import pytest
from datetime import date

from blog.models import Blog, Comment, BlogReview
from django.contrib.auth import get_user_model

User = get_user_model()


@pytest.fixture
def user(db):
    return User.objects.create_user("alice", "a@e.com", "pw")


@pytest.fixture
def blog(db):
    return Blog.objects.create(
        title="Refactoring",
        author="martinfowler",
        published_date=date(2026, 1, 10),
    )


@pytest.fixture
def comment(blog):
    return Comment.objects.create(blog=blog, content="one two three four")


@pytest.fixture
def review(blog, user):
    return BlogReview.objects.create(blog=blog, user=user, rating=5, comment="Hi")


In [ ]:
# blog/tests/test_models.py

import pytest
from django.db import IntegrityError
from django.core.exceptions import ValidationError
from datetime import date

from blog.models import Blog, BlogReview, CategoryType


class TestBlogModel:
    def test_str(self, blog):
        assert str(blog) == "Refactoring"

    def test_default_category(self, blog):
        assert blog.category_type == CategoryType.TECH

    def test_title_too_long(self, db):
        b = Blog(title="x" * 201, author="a", published_date=date(2026, 1, 1))
        with pytest.raises(ValidationError):
            b.full_clean()


class TestBlogReviewModel:
    def test_unique_per_user(self, blog, user):
        BlogReview.objects.create(blog=blog, user=user, rating=4)
        with pytest.raises(IntegrityError):
            BlogReview.objects.create(blog=blog, user=user, rating=2)

    def test_rating_negative_invalid(self, blog, django_user_model):
        u = django_user_model.objects.create_user("v", "v@e.com", "pw")
        r = BlogReview(blog=blog, user=u, rating=-1)
        with pytest.raises(ValidationError):
            r.full_clean()


<br>

## Summary

---

| Concept | Description | Example |
|---------|-------------|---------|
| **@pytest.mark.django_db** | Enable database access | `@pytest.mark.django_db` |
| **db fixture** | Enable database in fixtures | `def blog(db):` |
| **full_clean()** | Trigger model validation | `model.full_clean()` |
| **IntegrityError** | Database constraint violation | `pytest.raises(IntegrityError)` |
| **ValidationError** | Model validation failure | `pytest.raises(ValidationError)` |
| **Fixtures** | Reusable test data | `@pytest.fixture` |
| **Reverse relations** | Access related objects | `blog.comments.all()` |

<br>

**Key principles:**

1. **Use fixtures** for test data — keep tests short
2. **Test `__str__`** — admin and debugging depend on it
3. **Test validation** with `full_clean()` — `save()` does not validate automatically
4. **Test relationships** — forward FK and reverse related managers
5. **Test querysets** — filters, ordering, aggregates
6. **Put `db` on data fixtures** — avoids sprinkling markers on every test


<br>

### 🧠 EXERCISE 🧠, Test Blog __str__ and ordering

---


Write tests for the real **`Blog`** model (`mysite/blog/models.py`).

<br>

**Your task**

1. `__str__` returns the **title**.
2. Default **`category_type`** is **Technology** (`CategoryType.TECH`).
3. **`Meta.ordering`** is respected: newest `published_date` first, then `title` (create at least two rows with different dates).

Use `@pytest.mark.django_db` or the `db` fixture.


<details>
    <summary>▶️ Solution</summary>

```python
# blog/tests/test_models.py

import pytest
from datetime import date

from blog.models import Blog, CategoryType


@pytest.mark.django_db
def test_str_is_title():
    b = Blog.objects.create(title="A", author="a", published_date=date(2026, 1, 1))
    assert str(b) == "A"


@pytest.mark.django_db
def test_default_category():
    b = Blog.objects.create(title="B", author="b", published_date=date(2026, 1, 2))
    assert b.category_type == CategoryType.TECH


@pytest.mark.django_db
def test_ordering_by_published_date_then_title():
    Blog.objects.create(title="Z", author="z", published_date=date(2026, 1, 1))
    Blog.objects.create(title="A", author="a", published_date=date(2026, 2, 1))
    titles = list(Blog.objects.values_list("title", flat=True))
    assert titles[0] == "A"
    assert titles[1] == "Z"
```

</details>


<br>

### 🧠 EXERCISE 🧠, Test QuerySets on Blog

---


Write tests for **QuerySet** operations on **`Blog`** using only fields that exist in the course project.

<br>

**Your task**

1. `filter(category_type=CategoryType.BUSINESS)` returns only business posts.
2. `filter(author__icontains="martin")` is case-insensitive on the `author` **CharField**.
3. Combine `filter` + `order_by("title")` and assert the first row's title.

Create rows inside each test or with small fixtures.


<details>
    <summary>▶️ Solution</summary>

```python
# blog/tests/test_models.py

import pytest
from datetime import date

from blog.models import Blog, CategoryType


@pytest.mark.django_db
def test_filter_business():
    Blog.objects.create(title="Tech", author="a", published_date=date(2026, 1, 1), category_type=CategoryType.TECH)
    b = Blog.objects.create(title="Biz", author="b", published_date=date(2026, 1, 2), category_type=CategoryType.BUSINESS)
    qs = Blog.objects.filter(category_type=CategoryType.BUSINESS)
    assert list(qs) == [b]


@pytest.mark.django_db
def test_filter_author_icontains():
    Blog.objects.create(title="1", author="Alice", published_date=date(2026, 1, 1))
    Blog.objects.create(title="2", author="BOB", published_date=date(2026, 1, 2))
    qs = Blog.objects.filter(author__icontains="alice")
    assert qs.count() == 1


@pytest.mark.django_db
def test_filter_order_by_title():
    Blog.objects.create(title="Zed", author="z", published_date=date(2026, 1, 1))
    Blog.objects.create(title="Amy", author="a", published_date=date(2026, 1, 1))
    first = Blog.objects.filter(author__in=["z", "a"]).order_by("title").first()
    assert first.title == "Amy"
```

</details>


---